In [1]:
import sqlite3
import pandas as pd
import numpy as np
import pickle
import re
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

print("Libraries imported successfully")

Libraries imported successfully


# Tags Alterations

In [2]:
conn = sqlite3.connect('../data/tags_alterations_full_2025-10_v2.db')
df = pd.read_sql_query("SELECT * FROM tag_inconsistencies", conn)

print(f"Loaded {len(df)} tag inconsistencies from database")

KeyboardInterrupt: 

In [ ]:
df = df.drop(columns=['id'])

In [ ]:
for col in ['old_snap_timestamp', 'old_rev_timestamp', 'new_snap_timestamp', 'new_rev_timestamp', 'min_delta']:
    df[col] = pd.to_datetime(df[col], unit='s', errors='coerce')

In [ ]:
df['snap_year'] = df['new_snap_timestamp'].dt.year
df['rev_year'] = df['new_rev_timestamp'].dt.year
df['category'] = df['new_revision'].apply(lambda x: 'Move' if pd.notna(x) else 'Deletion')

In [ ]:
df['delta'] = np.where(
    (df['new_snapshot_cpt'] - df['old_snapshot_cpt']) < 2,
    0,
    (df['min_delta'] - df['old_snap_timestamp']).dt.days
)

In [ ]:
def categorize_platform(origin_url):
    if pd.isna(origin_url):
        return 'Other'
    if re.match(r'^https?://[^/]*github\.com', origin_url):
        return 'GitHub'
    elif re.match(r'^https?://[^/]*gitlab\.', origin_url):
        return 'GitLab'
    elif re.match(r'^https?://android\.googlesource\.com', origin_url):
        return 'Android'
    elif re.match(r'^https?://[^/]*bitbucket\.', origin_url):
        return 'BitBucket'
    elif re.match(r'^https?://[^/]*codeberg\.', origin_url):
        return 'Codeberg'
    elif re.match(r'^https?://[^/]*git\.sr\.ht', origin_url):
        return 'SourceHut'
    else:
        return 'Other'

df['platform'] = df['origin_url'].apply(categorize_platform)

In [ ]:
invalid_snapshots = df[df['old_snap_timestamp'] >= df['new_snap_timestamp']]

print(f"Total inconsistencies: {len(df)}")
print(f"Cases where old_snap_timestamp > new_snap_timestamp: {len(invalid_snapshots)}")
print(f"Valid snapshot ordering: {len(df) - len(invalid_snapshots)} ({100 * (len(df) - len(invalid_snapshots)) / len(df):.2f}%)")

if len(invalid_snapshots) > 0:
    print("\nSample of invalid cases:")
    print(invalid_snapshots[['origin_url', 'tag_name', 'old_snap_timestamp', 'new_snap_timestamp']].head(10))
else:
    print("\n✓ All old_snapshot timestamps are older than their corresponding new_snapshot timestamps")

Total inconsistencies: 15166153
Cases where old_snap_timestamp > new_snap_timestamp: 1
Valid snapshot ordering: 15166152 (100.00%)

Sample of invalid cases:
                                              origin_url          tag_name  \
11005277  https://github.com/cornerstonejs/cornerstone3D  refs/tags/v2.2.3   

          old_snap_timestamp  new_snap_timestamp  
11005277 2024-11-12 18:40:57 2024-11-12 18:40:57  


In [ ]:
print(f"length of df : {len(df):_}")
df.head(5)

length of df : 15_166_153


,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform
0,https://github.com/elevenlee/oop_generics,refs/tags/v1.0,lightweight,swh:1:snp:589750d264ea6ba72cd290d154b089ce79c20b14,0,2015-08-05 00:56:29,swh:1:rev:16f2c36c23dc1e432a183f2575be1e2b4d654ca9,2012-12-29 07:00:19,swh:1:dir:e482b7c2c58ab76031e45dd163c159e1a46b70a2,swh:1:snp:e613bc07696f35d8c1d341bd9928a313d3a705ac,1,2016-03-14 02:41:49,NaN,NaT,NaN,2015-08-05 00:56:29,2016,NaN,Deletion,0,GitHub
1,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/2.0.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:29e7fce9fc88a55bdafb507f6506bd6f153260b0,2020-04-27 03:35:14,swh:1:dir:c612010c922cea11ecdfa384db474123a0225d73,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub
2,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/2.2.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:16a7fefff1f1fb674d1ed0328099a8d6ac0d38bb,2020-04-27 19:31:38,swh:1:dir:3727b13bc24dad71024adebc98eb17a2b9310d35,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub
3,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/3.0.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:85401c9f014ff7d8950b1930e732b73fd6d08043,2020-05-07 07:24:36,swh:1:dir:65029e9180c14f7e3c5d0d1d3cf0b9b697b5e60e,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub
4,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/2.3.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:6fa41ccd9ece1292f0df3554e0c5f6680c82e818,2020-04-29 08:42:15,swh:1:dir:1edad0e47774c0fb05657616f6d8a4210cbcacb0,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub


## Stars

In [ ]:
stars = pd.read_pickle("/home/infres/rapaport/datasets/github-stars/github-stars.pkl")

In [ ]:
stars['origin'] = stars['origin'].str.strip('"')
stars = stars.drop_duplicates(subset='origin')

In [ ]:
df_bis = df.copy()

In [ ]:
df_bis = df_bis.merge(stars[['origin', 'stars']], 
              left_on='origin_url', 
              right_on='origin', 
              how='left')

print(f"DataFrame shape after merge: {df_bis.shape}")
print(f"Number of rows with stars data: {df_bis['stars'].notna().sum():_}")
print(f"Number of rows without stars data (null): {df_bis['stars'].isna().sum():_}")

DataFrame shape after merge: (70217, 23)
Number of rows with stars data: 70_217
Number of rows without stars data (null): 0


In [ ]:
df_bis = df_bis.drop(columns=['origin'])

In [ ]:
print(f"Initial: {len(df_bis):_}")
df_dd = df_bis.drop_duplicates().reset_index(drop=True)
print(f"Mid: {len(df_dd):_}")
df_min = df_bis[['origin_url', 'tag_name', 'type', 'old_snapshot', 'old_snap_timestamp']]
print(f"Init min {len(df_min):_}")
df_min_dd = df_min.drop_duplicates().reset_index(drop=True)
print(f"end: {len(df_min_dd):_}")

Initial: 70_217
Mid: 70_217
Init min 70_217
end: 70_217


In [ ]:
df[(df['origin_url']=="https://git.codelinaro.org/clo/la/platform/external/sepolicy.git") & (df['tag_name']=='refs/tags/android-4.4.1_r1') & (df['old_snapshot'] == "swh:1:snp:fa0ade8bea3fb0a1421f99889b42a0e913dedda7")]

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform


In [ ]:
with open("../data/tag_alterations_full_2025-10.pkl", "wb") as f:
    pickle.dump(df_bis, f)

In [14]:
df = pd.read_pickle("../data/tag_alterations_full_2025-10_v3.pkl")

In [18]:
df[(df['origin_url'].str.contains("changed-files")) & (df['new_revision'].str.contains('0e58ed8'))]

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform,stars
2198205,https://github.com/zendesk/changed-files,refs/tags/v18.7,lightweight,swh:1:snp:e49d3cef71c6999c581e52270685f5964c14f5bc,0,2024-09-20 23:18:29,swh:1:rev:a59f800cbb60ed483623848e31be67659a2940f8,2022-04-08 09:43:01,swh:1:dir:06e64979ae45fcd6a2a9b8b91b4a81cbcabe8e65,swh:1:snp:b2acad3bf9a7533de8360c346de440df7d859b92,9,2025-03-15 14:12:10,swh:1:rev:0e58ed8671d6b60d0890c21b07f8835ace038e67,2025-03-14 16:57:45,swh:1:dir:e7184c732b62349a7b7177a07fd80e82601f370a,2025-02-15 23:27:01,2025,2025.0,Alteration,148,GitHub,0.0
2198207,https://github.com/zendesk/changed-files,refs/tags/v37.2.0,lightweight,swh:1:snp:e49d3cef71c6999c581e52270685f5964c14f5bc,0,2024-09-20 23:18:29,swh:1:rev:68b429ddc666ea0dba46309e1ee45e06bb408df8,2023-07-18 17:41:01,swh:1:dir:ccab5fe2e97c81819606b28711e170e37904cc43,swh:1:snp:b2acad3bf9a7533de8360c346de440df7d859b92,9,2025-03-15 14:12:10,swh:1:rev:0e58ed8671d6b60d0890c21b07f8835ace038e67,2025-03-14 16:57:45,swh:1:dir:e7184c732b62349a7b7177a07fd80e82601f370a,2025-02-15 23:27:01,2025,2025.0,Alteration,148,GitHub,0.0
2198209,https://github.com/zendesk/changed-files,refs/tags/v18.4,lightweight,swh:1:snp:e49d3cef71c6999c581e52270685f5964c14f5bc,0,2024-09-20 23:18:29,swh:1:rev:e35d0afdc1f0b01f84ec0f4cdf1b179325634b36,2022-03-21 01:30:28,swh:1:dir:f4e749fa02bfc8a8a9ee4d5816586302a6d01619,swh:1:snp:b2acad3bf9a7533de8360c346de440df7d859b92,9,2025-03-15 14:12:10,swh:1:rev:0e58ed8671d6b60d0890c21b07f8835ace038e67,2025-03-14 16:57:45,swh:1:dir:e7184c732b62349a7b7177a07fd80e82601f370a,2025-02-15 23:27:01,2025,2025.0,Alteration,148,GitHub,0.0
2198211,https://github.com/zendesk/changed-files,refs/tags/v37.5.1,lightweight,swh:1:snp:e49d3cef71c6999c581e52270685f5964c14f5bc,0,2024-09-20 23:18:29,swh:1:rev:a96679dfee2a1e64b1db5a210c0ffaf1f2cb24ce,2023-07-28 19:34:23,swh:1:dir:63692aa7571c374eac46d4f60f044e7c67659fba,swh:1:snp:b2acad3bf9a7533de8360c346de440df7d859b92,9,2025-03-15 14:12:10,swh:1:rev:0e58ed8671d6b60d0890c21b07f8835ace038e67,2025-03-14 16:57:45,swh:1:dir:e7184c732b62349a7b7177a07fd80e82601f370a,2025-02-15 23:27:01,2025,2025.0,Alteration,148,GitHub,0.0
2198213,https://github.com/zendesk/changed-files,refs/tags/v34.0.0,lightweight,swh:1:snp:e49d3cef71c6999c581e52270685f5964c14f5bc,0,2024-09-20 23:18:29,swh:1:rev:c4d29bf5b2769a725bcc9a723c498ba9c34c05b4,2022-10-25 22:04:27,swh:1:dir:3940140177acbd1edeb44d119c2508ebae1f712f,swh:1:snp:b2acad3bf9a7533de8360c346de440df7d859b92,9,2025-03-15 14:12:10,swh:1:rev:0e58ed8671d6b60d0890c21b07f8835ace038e67,2025-03-14 16:57:45,swh:1:dir:e7184c732b62349a7b7177a07fd80e82601f370a,2025-02-15 23:27:01,2025,2025.0,Alteration,148,GitHub,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2198893,https://github.com/zendesk/changed-files,refs/tags/v43.0.1,lightweight,swh:1:snp:e49d3cef71c6999c581e52270685f5964c14f5bc,0,2024-09-20 23:18:29,swh:1:rev:20576b4b9ed46d41e2d45a2256e5e2316dde6834,2024-03-20 21:58:07,swh:1:dir:785398deebfb6e69eebd65156e7472f980c955de,swh:1:snp:b2acad3bf9a7533de8360c346de440df7d859b92,9,2025-03-15 14:12:10,swh:1:rev:0e58ed8671d6b60d0890c21b07f8835ace038e67,2025-03-14 16:57:45,swh:1:dir:e7184c732b62349a7b7177a07fd80e82601f370a,2025-02-15 23:27:01,2025,2025.0,Alteration,148,GitHub,0.0
2198895,https://github.com/zendesk/changed-files,refs/tags/v35.6.3,lightweight,swh:1:snp:e49d3cef71c6999c581e52270685f5964c14f5bc,0,2024-09-20 23:18:29,swh:1:rev:74b06cafc9658d2a91cc5ceb920fd6b5a5649051,2023-03-09 03:52:54,swh:1:dir:960881e12706eddd01ecd32998c2070f173fe14a,swh:1:snp:b2acad3bf9a7533de8360c346de440df7d859b92,9,2025-03-15 14:12:10,swh:1:rev:0e58ed8671d6b60d0890c21b07f8835ace038e67,2025-03-14 16:57:45,swh:1:dir:e7184c732b62349a7b7177a07fd80e82601f370a,2025-02-15

# Deletion --> Move

In [ ]:
conn = sqlite3.connect('../data/tags_alterations_full_2025-10_v2.db')
dm = pd.read_sql_query("SELECT * FROM deletion_creation_v2", conn)

print(f"Loaded {len(dm):_} tag inconsistencies from database")

Loaded 8_624 tag inconsistencies from database


In [ ]:
for col in ['old_snap_timestamp', 'new_snap_timestamp', 'creation_snap_ts', 'creation_rev_ts', 'creation_delta']:
    dm[col] = pd.to_datetime(dm[col], unit='s', errors='coerce')

In [ ]:
dm.head(2)

,origin_url,tag_name,type,old_snapshot,old_snap_timestamp,old_revision,old_root_dir,new_snapshot,new_snap_timestamp,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta
0,https://github.com/Semantic-Org/Semantic-UI,refs/tags/1.0.0-beta,lightweight,swh:1:snp:85c9d3964055f55be138e7422272687744e76e3c,2015-08-07 14:36:07,swh:1:rev:24c7baff8e097558390d96b3d588cd4d06ac1344,swh:1:dir:70534c3e6bfdb501ec71f7f33a1436bb0873a3d7,swh:1:snp:68ce2c47d3a215285e5e603c41677ca8236ac0a2,2016-03-13 23:16:32,annotated,swh:1:rev:24c7baff8e097558390d96b3d588cd4d06ac1344,2014-11-06 18:23:43,swh:1:dir:70534c3e6bfdb501ec71f7f33a1436bb0873a3d7,68ce2c47d3a215285e5e603c41677ca8236ac0a2,2016-03-13 23:16:32,2016-03-13 23:16:32
1,https://github.com/Semantic-Org/Semantic-UI,refs/tags/0.10.1,lightweight,swh:1:snp:85c9d3964055f55be138e7422272687744e76e3c,2015-08-07 14:36:07,swh:1:rev:005e58fed27e716fdbf7d6d778516d21dc138510,swh:1:dir:17987d6cfa990ab84707b170d709a5b3b13bf42c,swh:1:snp:68ce2c47d3a215285e5e603c41677ca8236ac0a2,2016-03-13 23:16:32,annotated,swh:1:rev:005e58fed27e716fdbf7d6d778516d21dc138510,2013-12-06 15:39:35,swh:1:dir:17987d6cfa990ab84707b170d709a5b3b13bf42c,68ce2c47d3a215285e5e603c41677ca8236ac0a2,2016-03-13 23:16:32,2016-03-13 23:16:32


In [ ]:
dm[dm['creation_delta']<dm['creation_snap_ts']]

,origin_url,tag_name,type,old_snapshot,old_snap_timestamp,old_revision,old_root_dir,new_snapshot,new_snap_timestamp,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta
364,https://github.com/Kong/kong,refs/tags/3.0.0,lightweight,swh:1:snp:8536a3e4ca6c826b8a517ab7a285e67547393f23,2022-08-16 18:20:28,swh:1:rev:a9e58a1f3c8bd2b89b1a7be67c9179d15bd06147,swh:1:dir:872f25ff43c261022a9fb980f7ff466e73fe0c36,swh:1:snp:d68e70525da41c5db514735dadf405d3e9f062a8,2022-08-29 01:01:26,annotated,swh:1:rev:78eeaa73ed84954952372dc40268c97ae66e8373,2022-09-12 12:38:55,swh:1:dir:205a27d78d9ccf75313adc652d7be0230f41f748,ba19ba3c074de5163890811f74cedf55a3dec1cf,2022-10-03 14:00:12,1970-01-06 02:03:17
1003,https://github.com/streamlit/streamlit,refs/tags/1.26.2,lightweight,swh:1:snp:fc0d9760b94019d6b876a9ba3b9de9da2d3a1625,2024-01-18 23:08:43,swh:1:rev:c5b02ba331f40c13f9d6fbb07b09b8ce6e4ac3d1,swh:1:dir:90052ca1f72f1c8d1cea505496cca1d9da90afb9,swh:1:snp:0ae89d6a40ae127e771dba58790801bfec2447a1,2024-09-24 08:58:44,annotated,swh:1:rev:6b415588354da764cae08ce1de3d975fee1513ec,2025-01-13 22:29:07,swh:1:dir:bccc2156b65c5d465850644bdb87daa4384b9f6c,48127677ecb256389e2928da4b362d9db545a4aa,2025-01-16 16:53:49,1970-03-28 22:50:18
1510,https://github.com/axios/axios,refs/tags/v1.2.2,annotated,swh:1:snp:fb801fdbb3d3ea14aed0acfc2bc167dc1c43b9ed,2022-12-21 02:23:28,swh:1:rev:3b6d5274aab55ce51cbf63ff04837a33a8bea985,swh:1:dir:2ba7928fad933cdb56d611025f681cf48df08adf,swh:1:snp:10cf3b21a3c05675d7474e64848b933890a9b30f,2022-12-25 12:06:43,lightweight,swh:1:rev:8ea432429b81c2f1aa8b03e43d0bdb498f21c4f4,2022-12-29 06:31:54,swh:1:dir:2d7311bc7ff266bd53097aebba8bd9e4258f8ab0,755b05718e6ddcf1e94190869df6172345a5f2bd,2023-01-08 03:12:02,1970-01-08 12:34:35
3358,https://github.com/psf/black,refs/tags/stable,lightweight,swh:1:snp:da93cd6d28416ecf7b01729d5ba94d7d47d652fa,2020-02-17 10:11:06,swh:1:rev:6bedb5c58a7d8c25aa9509f8217bc24e9797e90d,swh:1:dir:81e8ff9c02e1c08503cfd02340d8408e76164339,swh:1:snp:3fe667c6f2b58848627860b19486702243fae771,2020-12-12 12:38:59,annotated,swh:1:rev:d038a24ca200da9dacc1dcb05090c9e5b45b7869,2022-01-29 19:30:25,swh:1:dir:0da0ebbe109c9f879eec3801e165ce853cb1f3aa,0ac98da8352e6ec3907bde3fbfe227e3a8f90cc8,2022-02-13 14:36:27,1971-02-05 05:53:12
3764,https://github.com/scikit-learn/scikit-learn,refs/tags/debian/0.3-3,lightweight,swh:1:snp:74fb9f24390c67586452f3b526ee5177f7814b73,2015-07-29 04:50:18,swh:1:rev:47890ac823314f1a9e2920dff7575850af56c273,swh:1:dir:50126bbf3cc591649a421861ec6b99850968a319,swh:1:snp:8dee8f36e67d6b87803fd6f7fb8c55f979660a81,2016-03-03 00:20:11,annotated,swh:1:rev:47890ac823314f1a9e2920dff7575850af56c273,2010-05-08 16:48:33,swh:1:dir:50126bbf3cc591649a421861ec6b99850968a319,d6e3d6268442f00f39776e886274b3c808e75b05,2017-05-04 15:40:17,1970-07-24 09:48:33
3767,https://github.com/scikit-learn/scikit-learn,refs/tags/debian/0.5-1,lightweight,swh:1:snp:74fb9f24390c67586452f3b526ee5177f7814b73,2015-07-29 04:50:18,swh:1:rev:dc72677a9c13a656cda8be4b23cd897b56109b4b,swh:1:dir:55eb8c0af1f8d56746b0d52798b2812f43a5ba54,swh:1:snp:8dee8f36e67d6b87803fd6f7fb8c55f979660a81,2016-03-03 00:20:11,annotated,swh:1:rev:dc72677a9c13a656cda8be4b23cd897b56109b4b,2010-10-11 15:46:42,swh:1:dir:55eb8c0af1f8d56746b0d52798b2812f43a5ba54,d6e3d6268442f00f39776e886274b3c808e75b05,2017-05-04 15:40:17,1970-07-24 09:48:33
3771,https://github.com/scikit-learn/scikit-learn,refs/tags/debian/0.11.0-2,lightweight,swh:1:snp:74fb9f24390c67586452f3b526ee5177f7814b73,2015-07-29 04:50:18,swh:1:rev:114822b1e18c9d7f887c58b8a3b2c279bdce6d35,swh:1:dir:c989028cbc6d8632254dc4594bd2258ac7cca3e8,swh:1:snp:8dee8f36e67d6b87803fd6f7fb8c55f979660a81,2016-03-03 00:20:11,annotated,swh:1:rev:114822b1e18c9d7f887c58b8a3b2c279bdce6d35,2012-07-04 15:28:09,swh:1:dir:c989028cbc6d8632254dc4594bd2258ac7cca3e8,d6e3d6268442f00f39776e886274b3c808e75b05,2017-05-04 15:40:17,1970-07-24 09:48:33
3772,https://github.com/scikit-learn/scikit-learn,refs

In [ ]:
dm['deadtime'] = np.where(
    dm['creation_delta'] == dm['creation_snap_ts'],
    pd.Timedelta(0),
    np.where(
        dm['creation_delta'] < dm['creation_snap_ts'],
        dm['creation_delta'] - dm['new_snap_timestamp'],
        pd.NaT
    )
)

print(f"Created 'deadtime' column")
print(f"Deadtime = 0: {(dm['deadtime'] == pd.Timedelta(0)).sum():_}")
print(f"Deadtime calculated: {(dm['deadtime'].notna() & (dm['deadtime'] != pd.Timedelta(0))).sum():_}")
print(f"Deadtime = NA: {dm['deadtime'].isna().sum():_}")

Created 'deadtime' column
Deadtime = 0: 8_600
Deadtime calculated: 24
Deadtime = NA: 0


In [ ]:
dm['status'] = np.where(
    # (dm['creation_delta'] == dm['new_snap_timestamp']) & 
    (dm['type'] == "lightweight") & 
    (dm['creation_type'] == "annotated") &
    (dm['old_snap_timestamp'] < pd.Timestamp('2015-09-18')),
    'non-legit',
    'legit'
)

print(f"Status column created:")
print(f"Non-legit: {(dm['status'] == 'non-legit').sum():_}")
print(f"Legit: {(dm['status'] == 'legit').sum():_}")

Status column created:
Non-legit: 8_491
Legit: 133


In [ ]:
merge_keys = ['origin_url', 'tag_name', 'type', 'old_snapshot', 'old_snap_timestamp', 'old_revision', 'old_root_dir', 'new_snapshot', 'new_snap_timestamp']

dm_check = dm[merge_keys].drop_duplicates()
df_bis_check = df_bis[merge_keys].drop_duplicates()

print(f"Unique combinations in dm: {len(dm_check):_}")
print(f"Unique combinations in df_bis: {len(df_bis_check):_}")

dm_in_df_bis = dm_check.merge(df_bis_check, on=merge_keys, how='inner')
print(f"dm rows that match df_bis: {len(dm_in_df_bis):_} ({100*len(dm_in_df_bis)/len(dm_check):.2f}%)")

Unique combinations in dm: 8_624
Unique combinations in df_bis: 70_217
dm rows that match df_bis: 8_624 (100.00%)


In [ ]:
df_bis_merged = df_bis.merge(
    dm.drop(columns=['origin', 'stars'] if 'origin' in dm.columns and 'stars' in dm.columns else []),
    on=merge_keys,
    how='left',
    suffixes=('', '_dm')
)

print(f"\nOriginal df_bis shape: {df_bis.shape}")
print(f"Merged df_bis shape: {df_bis_merged.shape}")
print(f"Rows with dm data: {df_bis_merged['status'].notna().sum():_}")


Original df_bis shape: (70217, 22)
Merged df_bis shape: (70217, 31)
Rows with dm data: 8_624


In [ ]:
with open("../data/tags_alteration_full_2025-10_dm.pkl", "wb") as f:
    pickle.dump(df_bis_merged, f)

In [28]:
df_bis_merged[df_bis_merged['status'] == "legit"].sample(3)

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform,stars,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta,deadtime,status
14911,https://github.com/strapi/strapi,refs/tags/v5.5.0,annotated,swh:1:snp:a0af8cd01e29494e3c28e420634ff72cab379695,168,2024-12-06 10:02:48,swh:1:rev:9ec5695e0b96d927ff2c35e74b99377e00929d35,2024-12-04 14:21:16,swh:1:dir:c6a80f91b64eb596c864bd5f03472c5de7b88c84,swh:1:snp:261a9e04bda1e3946e6c857618fc7c597cdcadd1,169,2024-12-25 17:04:37,NaN,NaT,NaN,2024-12-06 10:02:48,2024,NaN,Deletion,0,GitHub,52449,lightweight,swh:1:rev:82460d7ac378cb4d41eb101e06a58b59fc567184,2024-12-04 15:15:06,swh:1:dir:5940dac959dfc481753aea0fab7068aa195f681b,261a9e04bda1e3946e6c857618fc7c597cdcadd1,2024-12-25 17:04:37,2024-12-25 17:04:37,0 days,legit
1539,https://github.com/pbatard/rufus,refs/tags/v2.9,lightweight,swh:1:snp:f1678631023698384e2eb31b5815e612c8267b5a,4,2016-05-29 06:53:23,swh:1:rev:e177144258183bb8f0f9d7669432c7a5467d0d41,2016-05-17 09:46:18,swh:1:dir:f6f524fb83a7c0b2bfedeebe1202580766ef4f69,swh:1:snp:d9e77e7a260beb685a176a291fdf77deab90d431,18,2018-10-21 11:09:36,NaN,NaT,NaN,2018-10-12 18:08:08,2018,NaN,Deletion,866,GitHub,20550,annotated,swh:1:rev:e177144258183bb8f0f9d7669432c7a5467d0d41,2016-05-17 09:46:18,swh:1:dir:f6f524fb83a7c0b2bfedeebe1202580766ef4f69,d9e77e7a260beb685a176a291fdf77deab90d431,2018-10-21 11:09:36,2018-10-21 11:09:36,0 days,legit
929,https://github.com/pbatard/rufus,refs/tags/v2.4,lightweight,swh:1:snp:2a3f4a168c40c5d2c6100b171d91bfdb6e0d1152,1,2016-03-12 15:57:36,swh:1:rev:ae06a39d2f646f216abb05908f6e8a6d6077620e,2015-09-27 19:01:38,swh:1:dir:5344d81c5870a3134a7d2288ecd37cfcf7c82c9b,swh:1:snp:d9e77e7a260beb685a176a291fdf77deab90d431,18,2018-10-21 11:09:36,NaN,NaT,NaN,2018-10-12 18:08:08,2018,NaN,Deletion,944,GitHub,20550,annotated,swh:1:rev:ae06a39d2f646f216abb05908f6e8a6d6077620e,2015-09-27 19:01:38,swh:1:dir:5344d81c5870a3134a7d2288ecd37cfcf7c82c9b,d9e77e7a260beb685a176a291fdf77deab90d431,2018-10-21 11:09:36,2018-10-21 11:09:36,0 days,legit


In [29]:
print(f"initial: {len(dm):_}")
dd = dm.drop_duplicates().reset_index(drop=True)
print(f"End: {len(dd):_}")

initial: 8_624
End: 8_624


In [30]:
with open("../data/deletion_move_teaser_2025-05.pkl", "wb") as f:
    pickle.dump(dm, f)

In [31]:
df_bis_merged[df_bis_merged['category']=="Alteration"]

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform,stars,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta,deadtime,status


In [32]:
df_bis_merged[(df_bis_merged['status']=="legit")]

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform,stars,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta,deadtime,status
144,https://github.com/pbatard/rufus,refs/tags/v2.11,lightweight,swh:1:snp:22ddf3b0203c5fe005fa3cda709a83e13d62aa86,11,2016-09-15 10:22:08,swh:1:rev:b3c76b1f61ce380461edf1f8dbccb93626d174da,2016-09-08 16:33:41,swh:1:dir:040bf4df959519df8349944f93fc82fd1cf4201a,swh:1:snp:d9e77e7a260beb685a176a291fdf77deab90d431,18,2018-10-21 11:09:36,NaN,NaT,NaN,2018-10-12 18:08:08,2018,NaN,Deletion,757,GitHub,20550,annotated,swh:1:rev:b3c76b1f61ce380461edf1f8dbccb93626d174da,2016-09-08 16:33:41,swh:1:dir:040bf4df959519df8349944f93fc82fd1cf4201a,d9e77e7a260beb685a176a291fdf77deab90d431,2018-10-21 11:09:36,2018-10-21 11:09:36,0 days 00:00:00,legit
208,https://github.com/pbatard/rufus,refs/tags/v2.13,lightweight,swh:1:snp:eb92a9a08253bfc9500445f5c28620f9ad586f52,13,2017-05-05 22:54:05,swh:1:rev:4670cfaf35e8c7bfbae87104410d5088b59e69b1,2017-04-06 10:50:31,swh:1:dir:e6d091c434d86fa7431b1fdf7cdc2c168974e9d6,swh:1:snp:d9e77e7a260beb685a176a291fdf77deab90d431,18,2018-10-21 11:09:36,NaN,NaT,NaN,2018-10-12 18:08:08,2018,NaN,Deletion,524,GitHub,20550,annotated,swh:1:rev:4670cfaf35e8c7bfbae87104410d5088b59e69b1,2017-04-06 10:50:31,swh:1:dir:e6d091c434d86fa7431b1fdf7cdc2c168974e9d6,d9e77e7a260beb685a176a291fdf77deab90d431,2018-10-21 11:09:36,2018-10-21 11:09:36,0 days 00:00:00,legit
469,https://github.com/pbatard/rufus,refs/tags/v3.1,lightweight,swh:1:snp:dfa299e6b4a91611b806dc8a67c9150bbef6fdb9,14,2018-09-12 10:54:30,swh:1:rev:faebe1040fd67472ee43218b5fbe3364e5dd0376,2018-06-19 11:00:28,swh:1:dir:3a0e1c96e792dca8cd15c9c637ddd71911a3dac0,swh:1:snp:d9e77e7a260beb685a176a291fdf77deab90d431,18,2018-10-21 11:09:36,NaN,NaT,NaN,2018-10-12 18:08:08,2018,NaN,Deletion,30,GitHub,20550,annotated,swh:1:rev:faebe1040fd67472ee43218b5fbe3364e5dd0376,2018-06-19 11:00:28,swh:1:dir:3a0e1c96e792dca8cd15c9c637ddd71911a3dac0,d9e77e7a260beb685a176a291fdf77deab90d431,2018-10-21 11:09:36,2018-10-21 11:09:36,0 days 00:00:00,legit
562,https://github.com/pbatard/rufus,refs/tags/v2.6,lightweight,swh:1:snp:2a3f4a168c40c5d2c6100b171d91bfdb6e0d1152,1,2016-03-12 15:57:36,swh:1:rev:b7d1b77e4fea72d16a8923f506778c49899b129e,2015-12-22 20:17:54,swh:1:dir:c3a832ff9c69d2e15e032f78b86e34fdcfe2adcc,swh:1:snp:d9e77e7a260beb685a176a291fdf77deab90d431,18,2018-10-21 11:09:36,NaN,NaT,NaN,2018-10-12 18:08:08,2018,NaN,Deletion,944,GitHub,20550,annotated,swh:1:rev:b7d1b77e4fea72d16a8923f506778c49899b129e,2015-12-22 20:17:54,swh:1:dir:c3a832ff9c69d2e15e032f78b86e34fdcfe2adcc,d9e77e7a260beb685a176a291fdf77deab90d431,2018-10-21 11:09:36,2018-10-21 11:09:36,0 days 00:00:00,legit
604,https://github.com/pbatard/rufus,refs/tags/v2.17,lightweight,swh:1:snp:dfa299e6b4a91611b806dc8a67c9150bbef6fdb9,14,2018-09-12 10:54:30,swh:1:rev:14d2a629c91350acb2859bddde201732a518627e,2017-09-12 09:48:10,swh:1:dir:6606e36a81c9706f003e50b0e2662a07e97452d8,swh:1:snp:d9e77e7a260beb685a176a291fdf77deab90d431,18,2018-10-21 11:09:36,NaN,NaT,NaN,2018-10-12 18:08:08,2018,NaN,Deletion,30,GitHub,20550,annotated,swh:1:rev:14d2a629c91350acb2859bddde201732a518627e,2017-09-12 09:48:10,swh:1:dir:6606e36a81c9706f003e50b0e2662a07e97452d8,d9e77e7a260beb685a176a291fdf77deab90d431,2018-10-21 11:09:36,2018-10-21 11:09:36,0 days 00:00:00,legit
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31185,https://github.com/metabase/metabase,refs/tags/nightly-ee,annotated,swh:1:snp:95c8235a950520a251a37d31abc8b83b6cb8d68d,157,2024-12-06 18:50:18,swh:1:rev:5bc320bbe768fff628e24e8ad1ec078f55997a28,2024-12-04 17:52:53,swh:1:dir:fe085ddd061850cae67e7827205609071934b0b8,swh:1:snp:5f81e3dcb3135

# Classification

In [3]:
conn = sqlite3.connect('../data/tags_alterations_teaser_2025-05.db')
df = pd.read_sql_query("SELECT * FROM tag_diffs", conn)

print(f"Loaded {len(df):_} rows from database")

Loaded 16_627 rows from database


In [14]:
df.head(5)

,origin_url,tag_name,tag_type,old_snapshot,altering_snapshot,old_snap_timestamp,old_release_swhid,new_release_swhid,old_revision_swhid,new_revision_swhid,old_directory_swhid,new_directory_swhid,rel_message_differs,rel_author_differs,rel_author_timestamp_differs,rev_message_differs,rev_author_differs,rev_committer_differs,rev_author_timestamp_differs,rev_committer_timestamp_differs,dir_added,dir_deleted,dir_modified,dir_renamed
0,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/updater,lightweight,swh:1:snp:e58a72c6009fe5e33b7eaf48fb516814d86ccb18,swh:1:snp:63bfdf73657a8404a7d26a80c30453a01afeb345,1701946964,NaN,NaN,swh:1:rev:66340a27fa66cb2b42e008e5b4836c96ef1e05dd,swh:1:rev:f6b9ccc774fb728733555d90b27e5908aa1a41e1,swh:1:dir:9fc00eed989876ae112dfa1f991339ab25d4f1f7,swh:1:dir:e2bfa706d1a1b0b91c0e217dc26e52926a46ee2b,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,243.0,37.0,47.0,0.0
1,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:e58a72c6009fe5e33b7eaf48fb516814d86ccb18,swh:1:snp:71ed1f1c7b8a1bcfea6f0d4681a6ea446f028463,1701946964,NaN,NaN,swh:1:rev:baa3e15702b9cfa94dcfd77fc20c81ba4a77ad5f,swh:1:rev:9b79da2cdf82fdc08b262bdc697e074220ab4db1,swh:1:dir:e524468144a75d0f511acfb6bfbe87fc49293da4,swh:1:dir:2b69e3776f81e012549e15e5e67e26393cdf252f,NaN,NaN,NaN,1.0,1.0,0.0,1.0,1.0,117.0,48.0,74.0,0.0
2,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:71ed1f1c7b8a1bcfea6f0d4681a6ea446f028463,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1710458148,NaN,NaN,swh:1:rev:9b79da2cdf82fdc08b262bdc697e074220ab4db1,swh:1:rev:3dbb71a076fd9b0d7fab3d3102fe37280e0eff31,swh:1:dir:2b69e3776f81e012549e15e5e67e26393cdf252f,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,2.0,1.0,39.0,0.0
3,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1711481910,NaN,NaN,swh:1:rev:3dbb71a076fd9b0d7fab3d3102fe37280e0eff31,swh:1:rev:2ec841eb610e8d525525d897960cdc023d1324d7,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,NaN,NaN,NaN,1.0,0.0,0.0,1.0,1.0,3.0,1.0,33.0,0.0
4,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,swh:1:snp:da87c3a3e5985dfb44b14bda8f41ab549b5a912b,1712504281,NaN,NaN,swh:1:rev:2ec841eb610e8d525525d897960cdc023d1324d7,swh:1:rev:32212a46e205def8254de32c2909e2697e45b768,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,swh:1:dir:f7cb36c879c73a9a3eeded0dc9a96f36f1faacd0,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,43.0,16.0,139.0,0.0


In [15]:
conn = sqlite3.connect('../data/tags_alterations_teaser_2025-05.db')
files = pd.read_sql_query("SELECT * FROM tag_file_diffs", conn)

print(f"Loaded {len(files):_} rows from database")

Loaded 14_612_360 rows from database


In [16]:
files.head(3)

,origin_url,tag_name,tag_type,old_snapshot,old_snap_timestamp,altering_snapshot,new_snap_timestamp,old_directory_swhid,new_directory_swhid,file_path,change_type
0,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/updater,lightweight,swh:1:snp:e58a72c6009fe5e33b7eaf48fb516814d86ccb18,1701946964,swh:1:snp:63bfdf73657a8404a7d26a80c30453a01afeb345,1742181017,swh:1:dir:9fc00eed989876ae112dfa1f991339ab25d4f1f7,swh:1:dir:e2bfa706d1a1b0b91c0e217dc26e52926a46ee2b,src/assets/fonts/Twemoji.Mozilla.ttf,added
1,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/updater,lightweight,swh:1:snp:e58a72c6009fe5e33b7eaf48fb516814d86ccb18,1701946964,swh:1:snp:63bfdf73657a8404a7d26a80c30453a01afeb345,1742181017,swh:1:dir:9fc00eed989876ae112dfa1f991339ab25d4f1f7,swh:1:dir:e2bfa706d1a1b0b91c0e217dc26e52926a46ee2b,src/assets/image/itemicon/profiles.svg,added
2,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/updater,lightweight,swh:1:snp:e58a72c6009fe5e33b7eaf48fb516814d86ccb18,1701946964,swh:1:snp:63bfdf73657a8404a7d26a80c30453a01afeb345,1742181017,swh:1:dir:9fc00eed989876ae112dfa1f991339ab25d4f1f7,swh:1:dir:e2bfa706d1a1b0b91c0e217dc26e52926a46ee2b,src/components/layout/layout-control.tsx,added


In [17]:
files[(files['origin_url'] == 'https://github.com/clash-verge-rev/clash-verge-rev') & (files['tag_name'] == 'refs/tags/alpha') & (files['tag_type'] == 'lightweight') & (files['old_snapshot'] == 'swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523')]

,origin_url,tag_name,tag_type,old_snapshot,old_snap_timestamp,altering_snapshot,new_snap_timestamp,old_directory_swhid,new_directory_swhid,file_path,change_type
608,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1711481910,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1712504281,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,src-tauri/src/core/service.rs,added
609,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1711481910,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1712504281,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,src-tauri/src/utils/unix_helper.rs,added
610,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1711481910,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1712504281,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,src/components/base/base-styled-text-field.tsx,added
611,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1711481910,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1712504281,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,src-tauri/src/core/win_service.rs,deleted
612,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1711481910,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1712504281,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,.gitignore,modified
613,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1711481910,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1712504281,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,UPDATELOG.md,modified
614,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1711481910,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1712504281,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,package.json,modified
615,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1711481910,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1712504281,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,src-tauri/Cargo.lock,modified
616,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1711481910,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1712504281,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,src-tauri/Cargo.toml,modified
617,https://github.com/clash-verge-rev/clash-verge-rev,refs/tags/alpha,lightweight,swh:1:snp:ba470f9a60e00d25da2632a037472ee7d0f27523,1711481910,swh:1:snp:39eaf06fdaa7f462d0a98766e504edf2ce38e3f2,1712504281,swh:1:dir:ee6b9a3ca17cc08521b8c31f41b444081f60acd2,swh:1:dir:fa065fa10384757eded74d24b31c8b73e06bae62,src-tauri/tauri.conf.json,modified


# Nix-Correlations

In [19]:
conn = sqlite3.connect('../data/tags_alterations_full_2025-10_v2.db')
#conn = sqlite3.connect('../data/tags_alterations_teaser_2025-05.db')
nix = pd.read_sql_query("SELECT * FROM nix_correlations", conn)

print(f"Loaded {len(nix):_} rows from database")

Loaded 4_719 rows from database


In [20]:
nix.head(5)

,origin_url,nix_file_path
0,https://github.com/rwos/gti,/swh/scratch/rapaport/nixpkgs/pkgs/by-name/gt/gti/package.nix
1,https://github.com/BinaryAnalysisPlatform/bap,/swh/scratch/rapaport/nixpkgs/pkgs/development/ocaml-modules/bap/default.nix
2,https://github.com/BinaryAnalysisPlatform/bap,/swh/scratch/rapaport/nixpkgs/pkgs/development/python-modules/bap/default.nix
3,https://github.com/grml/grml-etc-core,/swh/scratch/rapaport/nixpkgs/pkgs/by-name/gr/grml-zsh-config/package.nix
4,https://github.com/linuxmint/cinnamon-menus,/swh/scratch/rapaport/nixpkgs/pkgs/by-name/ci/cinnamon-menus/package.nix
